# Experimental circular-finned-tube examples

This executed notebook exercises the dry circular-finned-tube branch without optional property backends. It compares a welded constant-thickness fin (`D_root = D_o`) with an extruded linearly tapered fin (`D_root > D_o`). A continuous spiral fin is represented by the model as an equivalent periodic train of full annular fins. All calculations use SI internally.

In [1]:
import math
from pathlib import Path
import sys

repo_root = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / 'core').is_dir() and (path / 'pyproject.toml').is_file()
)
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from core.geometry import BareTube, CircularFinnedTube, TubeBundle
from core.heat_transfer import (
    annular_fin_efficiency,
    calculate_finned_tube_outside_heat_transfer,
)
from core.models import BareTubeHeatExchanger, BalanceSideSpec, HXSideInput
from core.pressure_drop import calculate_finned_tube_bank_hydraulics
from core.properties.common import FluidTransportProperties
from core.properties.fluids import ConstantPropertyProvider

INCH = 0.0254
FOOT = 0.3048
LBM = 0.45359237
BTU = 1055.05585262
P = 101_325.0

# Air state used in a published handbook-style Briggs--Young /
# Robinson--Briggs benchmark.
rho_o = 0.0765 * LBM / FOOT**3
mu_o = 0.0439 * LBM / (FOOT * 3600.0)
k_o = 0.0150 * BTU / (3600.0 * FOOT * (5.0 / 9.0))
Pr_o = 0.702
cp_o = Pr_o * k_o / mu_o
outside_props = FluidTransportProperties(rho_o, mu_o, k_o, cp_o)
outside_provider = ConstantPropertyProvider(outside_props)
inside_props = FluidTransportProperties(1.2, 1.9e-5, 0.028, 1050.0)
inside_provider = ConstantPropertyProvider(inside_props)

print(f'Outside benchmark state: rho={rho_o:.6f} kg/m3, mu={mu_o:.8g} Pa s, Pr={Pr_o:.3f}')

Outside benchmark state: rho=1.225412 kg/m3, mu=1.8147333e-05 Pa s, Pr=0.702


In [2]:
def make_bundle(*, tapered):
    D_root = 0.750 * INCH
    core = BareTube(
        D_i=0.600 * INCH,
        D_o=(0.700 if tapered else 0.750) * INCH,
        length_total=FOOT,
        length_effective=FOOT,
        wall_k=45.0,
    )
    tube = CircularFinnedTube(
        core_tube=core,
        fin_k=200.0,
        D_fin=1.625 * INCH,
        D_root=D_root,
        fin_thickness_root=(0.024 if tapered else 0.019) * INCH,
        fin_thickness_tip=(0.014 if tapered else 0.019) * INCH,
        fin_pitch=INCH / 9.0,
        fin_contact_resistance=0.0,
    )
    P_t = 1.875 * INCH
    return TubeBundle(
        tube=tube, n_rows=6, n_tubes_per_row=4,
        pitch_transverse=P_t,
        pitch_longitudinal=math.sqrt(3.0) * P_t / 2.0,
        layout='staggered', n_passes_tube=1,
        flow_arrangement='crossflow',
    )

bundles = {
    'welded constant fin': make_bundle(tapered=False),
    'extruded tapered fin': make_bundle(tapered=True),
}

for name, bundle in bundles.items():
    tube = bundle.tube
    print(f'\n{name}')
    print(f'  D_o / D_root / D_fin [mm] : {1e3*tube.D_o:.3f} / {1e3*tube.D_root:.3f} / {1e3*tube.D_fin:.3f}')
    print(f'  t_root / t_tip [mm]       : {1e3*tube.fin_thickness_root:.4f} / {1e3*tube.fin_thickness_tip_effective:.4f}')
    print(f'  effective fin count       : {tube.effective_fin_count:.6f}')
    print(f'  A_primary / A_fin [m2]    : {bundle.total_primary_outside_area:.6f} / {bundle.total_fin_area:.6f}')
    print(f'  A_gross / A_geometric [m2]: {bundle.total_outer_area:.6f} / {bundle.total_outer_geometric_area:.6f}')
    print(f'  fin volume per length     : {tube.fin_volume_per_length:.8g} m3/m')
    print(f'  periodic blockage         : {1e3*bundle.projected_blocking_area_per_length:.4f} mm')
    print(f'  face / minimum area [m2]  : {bundle.frontal_flow_area:.6f} / {bundle.minimum_free_flow_area:.6f}')


welded constant fin
  D_o / D_root / D_fin [mm] : 19.050 / 19.050 / 41.275
  t_root / t_tip [mm]       : 0.4826 / 0.4826
  effective fin count       : 108.000000
  A_primary / A_fin [m2]    : 0.362932 / 5.620963
  A_gross / A_geometric [m2]: 5.983895 / 5.983895
  fin volume per length     : 0.00018006325 m3/m
  periodic blockage         : 22.8505 mm
  face / minimum area [m2]  : 0.058064 / 0.030205

extruded tapered fin
  D_o / D_root / D_fin [mm] : 17.780 / 19.050 / 41.275
  t_root / t_tip [mm]       : 0.6096 / 0.3556
  effective fin count       : 108.000000
  A_primary / A_fin [m2]    : 0.343231 / 5.578634
  A_gross / A_geometric [m2]: 5.921866 / 5.921866
  fin volume per length     : 0.00017424403 m3/m
  periodic blockage         : 22.8505 mm
  face / minimum area [m2]  : 0.058064 / 0.030205


In [3]:
# Set the approach velocity to the 600 ft/min benchmark value.
V_face = 600.0 * FOOT / 60.0

for name, bundle in bundles.items():
    m_dot_outside = rho_o * V_face * bundle.frontal_flow_area
    ht = calculate_finned_tube_outside_heat_transfer(
        m_dot_outside, bundle, outside_props
    )
    hydraulic = calculate_finned_tube_bank_hydraulics(
        m_dot_outside, bundle, inlet_props=outside_props,
        temperature_in=420.0, temperature_out=420.0, pressure=P,
    )
    eta = annular_fin_efficiency(bundle.tube, ht.alpha)
    print(f'\n{name}')
    print(f'  V_face / V_ref [m/s] : {ht.face_velocity:.6f} / {ht.reference_velocity:.6f}')
    print(f'  Re_Droot / j / Nu    : {ht.Re:.6f} / {ht.j:.9f} / {ht.Nu:.6f}')
    print(f'  physical outside HTC : {ht.alpha:.6f} W/(m2 K)')
    print(f'  eta_fin / eta_overall: {eta.fin_efficiency:.8f} / {eta.overall_surface_efficiency:.8f}')
    print(f'  f_RB / delta_p       : {hydraulic.midpoint.f:.9f} / {hydraulic.dp_total:.6f} Pa')
    print(f'  warning count        : {len(set(w.code for w in (*ht.warnings, *hydraulic.warnings)))}')


welded constant fin
  V_face / V_ref [m/s] : 3.048000 / 5.859285
  Re_Droot / j / Nu    : 7537.183438 / 0.006802745 / 45.569298
  physical outside HTC : 62.101074 W/(m2 K)
  eta_fin / eta_overall: 0.92476684 / 0.92932985
  f_RB / delta_p       : 0.241008389 / 121.670396 Pa
  warning count        : 3

extruded tapered fin
  V_face / V_ref [m/s] : 3.048000 / 5.859285
  Re_Droot / j / Nu    : 7537.183438 / 0.006802745 / 45.569298
  physical outside HTC : 62.101074 W/(m2 K)
  eta_fin / eta_overall: 0.93325708 / 0.93712550
  f_RB / delta_p       : 0.241008389 / 121.670396 Pa
  warning count        : 5


In [4]:
simulations = {}
for name, bundle in bundles.items():
    m_dot_outside = rho_o * V_face * bundle.frontal_flow_area
    hx = BareTubeHeatExchanger(bundle)
    simulation = hx.simulate(
        HXSideInput(provider=inside_provider, m_dot=0.08, T_in=300.0, p=P),
        HXSideInput(provider=outside_provider, m_dot=m_dot_outside, T_in=420.0, p=P),
    )
    simulations[name] = (hx, m_dot_outside, simulation)
    d = simulation.finned_tube_diagnostics
    print(f'\n{name} -- full Simulation')
    print(f'  converged / iterations : {simulation.converged} / {simulation.iterations}')
    print(f'  Q / UA                 : {simulation.q:.6f} W / {simulation.UA:.6f} W/K')
    print(f'  T_inside,out / T_outside,out: {simulation.T_out_inside:.6f} K / {simulation.T_out_outside:.6f} K')
    print(f'  h_o / eta_fin          : {d.outside_htc:.6f} / {d.fin_efficiency:.8f}')
    print(f'  A_gross / A_effective  : {d.A_outside_gross:.6f} / {d.A_outside_effective:.6f} m2')
    print(f'  U[gross] / delta_p     : {d.U:.6f} W/(m2 K) / {d.dp:.6f} Pa')
    print(f'  correlation methods    : {d.heat_transfer_metadata.method} / {d.pressure_drop_metadata.method}')


welded constant fin -- full Simulation
  converged / iterations : True / 10
  Q / UA                 : 2433.099373 W / 24.542954 W/K
  T_inside,out / T_outside,out: 328.965469 K / 408.828626 K
  h_o / eta_fin          : 62.101074 / 0.92476684
  A_gross / A_effective  : 5.983895 / 5.561012 m2
  U[gross] / delta_p     : 4.101501 W/(m2 K) / 121.670396 Pa
  correlation methods    : briggs_young_1963 / robinson_briggs_1966

extruded tapered fin -- full Simulation
  converged / iterations : True / 10
  Q / UA                 : 2433.995968 W / 24.553990 W/K
  T_inside,out / T_outside,out: 328.976142 K / 408.824509 K
  h_o / eta_fin          : 62.101074 / 0.93325708
  A_gross / A_effective  : 5.921866 / 5.549531 m2
  U[gross] / delta_p     : 4.146327 W/(m2 K) / 121.670396 Pa
  correlation methods    : briggs_young_1963 / robinson_briggs_1966


In [5]:
# Close a Rating at the welded case's achieved Simulation temperature program.
hx, m_dot_outside, simulation = simulations['welded constant fin']
rating = hx.rate(
    BalanceSideSpec(
        provider=inside_provider, p=P, m_dot=0.08,
        T_in=300.0, T_out=simulation.T_out_inside,
    ),
    BalanceSideSpec(
        provider=outside_provider, p=P, m_dot=m_dot_outside,
        T_in=420.0, T_out=simulation.T_out_outside,
    ),
)
print('Welded constant fin -- full Rating')
print(f'  Q_required / UA_actual : {rating.Q_required:.6f} W / {rating.UA_actual:.6f} W/K')
print(f'  U_mean (gross area)     : {rating.U_mean:.6f} W/(m2 K)')
print(f'  overdesign / UA margin  : {rating.overdesign_factor:.9g} / {rating.ua_margin:.9g}')
print(f'  outside delta_p         : {rating.finned_tube_diagnostics.dp:.6f} Pa')

Welded constant fin -- full Rating
  Q_required / UA_actual : 2433.099373 W / 24.542954 W/K
  U_mean (gross area)     : 4.101501 W/(m2 K)
  overdesign / UA margin  : 6.66133815e-16 / 6.66133815e-16
  outside delta_p         : 121.670396 Pa


The empirical HTC and pressure-loss calculations use `D_root`, mass flux through the periodic minimum free-flow area, and the original source coefficient definitions. The physical film coefficient is not multiplied by fin efficiency: the solver instead uses `A_primary + eta_fin A_fin`. The built-in models are dry-only and reject inline banks and an activated wet/condensing path; source-range warnings remain visible and require engineering review.